# LeetCode 1058 — Minimize Rounding Error to Meet Target bằng Python

Link đề: https://leetcode.com/problems/minimize-rounding-error-to-meet-target/ · Độ khó: **Medium** · Premium (cần tài khoản LeetCode Premium để xem đề) · Chủ đề: Greedy (tham lam), Array (mảng), Math (toán), String (chuỗi), Sorting (sắp xếp)

> **Lưu ý:** đề bài của bài Premium này được viết theo hiểu biết và theo bản mô tả đã có; nên đối chiếu lại nếu bạn có tài khoản Premium.

Skeleton LeetCode:

```python
class Solution:
    def minimizeError(self, prices: List[str], target: int) -> str:
        ...
```

Ghi chú: cho mảng giá `prices` (mỗi giá là chuỗi số thực đúng `3` chữ số thập phân) và một số nguyên `target`. Mỗi giá được làm tròn **xuống** (`floor`) hoặc **lên** (`ceil`) sao cho tổng các số đã làm tròn đúng bằng `target`. Nếu không thể thì trả về chuỗi `"-1"`; ngược lại trả về **sai số làm tròn nhỏ nhất** `Σ |Round_i(p_i) - p_i|` dưới dạng chuỗi có đúng `3` chữ số thập phân. Ý tưởng chính: tổng số lần làm tròn lên bị ép bằng `d = target - (tổng các floor)`; mỗi giá có phần lẻ `f` thì làm tròn xuống tốn `f`, làm tròn lên tốn `1 - f`, nên **làm tròn lên cho `d` giá có phần lẻ lớn nhất**. Để tránh sai số số thực, ta làm việc bằng **số nguyên** (đơn vị `0.001`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề và vì sao tổng số lần làm tròn lên bị **ép** đúng bằng `d = target - floor_sum`.
2. Nắm các công cụ Python cần thiết: tách chuỗi `split("." )`, chuyển sang số nguyên đơn vị `0.001`, định dạng `f"{x // 1000}.{x % 1000:03d}"`, và (để kiểm chứng) `fractions.Fraction`, `decimal.Decimal`.
3. Hiểu **bất biến và hoán đổi (exchange)**: đổi một giá từ "xuống" sang "lên" thêm đúng `1000 - 2f` vào sai số, nên chọn các giá có `f` lớn nhất để tăng ít nhất.
4. Đi qua 4 cách: thử mọi cách làm tròn bằng `Fraction` (brute force, dùng làm oracle), quy hoạch động theo số lần làm tròn lên, tham lam sắp xếp phần lẻ giảm dần (cách nộp chính) và đếm phần lẻ theo từng giá trị `0..999`.
5. Tránh các lỗi kinh điển: dùng số thực rồi cắt bớt chữ số, quên định dạng đúng `3` chữ số, làm tròn lên cho giá có phần lẻ **nhỏ** nhất, coi số nguyên cũng làm tròn lên được, và thiếu kiểm tra cận dưới `target < floor_sum`.

## 2. Đề bài & ví dụ

Cho mảng `prices = [p_1, p_2, ..., p_n]` và `target`. Làm tròn mỗi `p_i` thành `Round_i(p_i)` sao cho mảng sau khi làm tròn có tổng bằng `target`. Mỗi phép `Round_i(p_i)` là `Floor(p_i)` hoặc `Ceil(p_i)`.

Trả về chuỗi `"-1"` nếu không thể làm cho tổng bằng `target`. Ngược lại trả về **sai số làm tròn nhỏ nhất**, định nghĩa là `Σ |Round_i(p_i) - p_i|` với `i` từ `1` đến `n`, dưới dạng chuỗi có **ba chữ số sau dấu thập phân**.

**Ví dụ 1:** `prices = ["0.700","2.800","4.900"]`, `target = 8` -> `"1.000"`. Dùng Floor, Ceil, Ceil: `(0.7 - 0) + (3 - 2.8) + (5 - 4.9) = 0.7 + 0.2 + 0.1 = 1.0`.

**Ví dụ 2:** `prices = ["1.500","2.500","3.500"]`, `target = 10` -> `"-1"` (không thể đạt tổng 10).

**Ví dụ 3:** `prices = ["1.500","2.500","3.500"]`, `target = 9` -> `"1.500"`.

**Ràng buộc:**

- `1 <= prices.length <= 500`
- Mỗi `prices[i]` biểu diễn một số thực trong đoạn `[0.0, 1000.0]` với đúng `3` chữ số thập phân.
- `0 <= target <= 10^6`

In [1]:
from typing import List, Optional
from fractions import Fraction
from decimal import Decimal
from itertools import product, combinations_with_replacement
import random, time

examples = [
    (["0.700", "2.800", "4.900"], 8, "1.000"),
    (["1.500", "2.500", "3.500"], 10, "-1"),
    (["1.500", "2.500", "3.500"], 9, "1.500"),
]
for prices, target, expected in examples:
    print(f"prices={prices} target={target} -> kỳ vọng {expected!r}")

prices=['0.700', '2.800', '4.900'] target=8 -> kỳ vọng '1.000'
prices=['1.500', '2.500', '3.500'] target=10 -> kỳ vọng '-1'
prices=['1.500', '2.500', '3.500'] target=9 -> kỳ vọng '1.500'


## 3. Kiến thức nền

### 3.1. Tại sao tránh số thực: dùng số nguyên đơn vị `0.001`

Máy tính lưu số thực nhị phân nên `0.7 + 0.2 + 0.1` không đúng bằng `1.0`. Mỗi giá có đúng `3` chữ số thập phân nên ta đọc chuỗi `"a.bcd"` thành số nguyên `a * 1000 + bcd` (đơn vị `0.001`) và chỉ làm việc với số nguyên.

In [2]:
print(0.7 + 0.2 + 0.1, 0.7 + 0.2 + 0.1 == 1.0)
print(Fraction("0.700") + Fraction("0.200") + Fraction("0.100") == 1)
print(Decimal("0.700") + Decimal("0.200") + Decimal("0.100"))
whole, dec = "4.900".split(".")
print(whole, dec, int(whole) * 1000 + int(dec))

0.9999999999999999 False
True
1.000
4 900 4900


### 3.2. Phần nguyên và phần lẻ

Với giá `p` (đơn vị `0.001`, số nguyên `v`): `floor = v // 1000`, phần lẻ `f = v % 1000`. Nếu `f == 0` thì `ceil = floor`; nếu `f > 0` thì `ceil = floor + 1`. Sai số khi làm tròn xuống là `f` (đơn vị `0.001`), khi làm tròn lên là `1000 - f`.

In [3]:
for s in ["0.700", "2.800", "5.000", "999.999"]:
    whole, dec = s.split(".")
    v = int(whole) * 1000 + int(dec)
    fl, f = divmod(v, 1000)
    print(f"{s}: floor={fl}, f={f}, sai số xuống={f}, sai số lên={(1000 - f) if f else 0}")

0.700: floor=0, f=700, sai số xuống=700, sai số lên=300
2.800: floor=2, f=800, sai số xuống=800, sai số lên=200
5.000: floor=5, f=0, sai số xuống=0, sai số lên=0
999.999: floor=999, f=999, sai số xuống=999, sai số lên=1


### 3.3. Định dạng chuỗi

`f"{err // 1000}.{err % 1000:03d}"` ghép phần nguyên và phần lẻ đệm số `0` đủ `3` chữ số (`:03d`). Tránh `str(round(x, 3))` vì cho `"1.0"` thay vì `"1.000"`.

In [4]:
err = 1000
print(f"{err // 1000}.{err % 1000:03d}", f"{1500 // 1000}.{1500 % 1000:03d}", f"{7 // 1000}.{7 % 1000:03d}")
print(str(round(1.0, 3)), f"{1.0:.3f}")

1.000 1.500 0.007
1.0 1.000


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `prices = ["0.700","2.800","4.900"]`, `target = 8`.

- Phần nguyên (floor): `0 + 2 + 4 = 6`. Cả ba giá có phần lẻ khác `0` nên mỗi giá có thể "tăng thêm 1" nếu làm tròn lên.
- Cần tổng `8`, nên phải làm tròn lên đúng `d = 8 - 6 = 2` giá (và xuống `1` giá).
- Phần lẻ: `0.7`, `0.8`, `0.9`. Làm tròn xuống tốn `f`, làm tròn lên tốn `1 - f`.
- Chọn `2` giá làm tròn lên: `1 - f` nhỏ nhất khi `f` lớn nhất ⇒ chọn `0.9` và `0.8`; giá `0.7` làm tròn xuống.
- Sai số: `0.7 (xuống) + 0.2 (lên) + 0.1 (lên) = 1.0` ⇒ `"1.000"` ✓.

Ví dụ 2: tổng floor `1 + 2 + 3 = 6`, tối đa tăng thêm `3` ⇒ khoảng đạt được `[6, 9]`; `target = 10` nằm ngoài ⇒ `"-1"`.

## 5. Cách 1 — Brute Force: thử mọi cách làm tròn bằng `Fraction`

**Ý tưởng.** Mỗi giá có hai lựa chọn (xuống hoặc lên): `2^n` cách. Với mỗi cách, tính tổng đã làm tròn; nếu bằng `target` thì tính sai số bằng `Fraction` (chính xác tuyệt đối) và lấy nhỏ nhất. Đúng nguyên văn đề, không dùng phân tích tham lam nên là **oracle độc lập**.

In [5]:
import math

def fmt(err: Fraction) -> str:
    """Định dạng một Fraction là bội của 0.001 thành chuỗi 3 chữ số thập phân."""
    thousandths = err * 1000
    assert thousandths.denominator == 1
    x = int(thousandths)
    return f"{x // 1000}.{x % 1000:03d}"

def minimize_brute(prices: List[str], target: int) -> str:
    ps = [Fraction(p) for p in prices]
    best = None
    for choice in product((0, 1), repeat=len(ps)):             # 0: floor, 1: ceil
        total = 0
        err = Fraction(0)
        for p, c in zip(ps, choice):
            r = math.ceil(p) if c else math.floor(p)
            total += r
            err += abs(r - p)
        if total == target and (best is None or err < best):
            best = err
    return "-1" if best is None else fmt(best)

for prices, target, expected in examples:
    got = minimize_brute(prices, target)
    print(f"brute = {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

brute = '1.000' OK
brute = '-1' OK
brute = '1.500' OK


**Độ phức tạp và nút thắt.** `2^n` cách, mỗi cách `O(n)` ⇒ `O(n * 2^n)`; với `n = 500` là không thể. Đo thử:

In [6]:
rng = random.Random(1058)
for n in (8, 10, 12, 14):
    prices = [f"{rng.randint(0, 9)}.{rng.randint(1, 999):03d}" for _ in range(n)]
    base = sum(int(p.split(".")[0]) for p in prices)
    t0 = time.perf_counter()
    minimize_brute(prices, base + n // 2)
    dt = time.perf_counter() - t0
    print(f"n={n:2}: {dt * 1000:9.1f} ms")

n= 8:       4.4 ms
n=10:      20.7 ms
n=12:      94.8 ms


n=14:     639.4 ms


Nút thắt: ta xét nhiều cách chọn mà tổng không bằng `target`. Số lần làm tròn lên thực ra **bị ép** chính xác bằng một số `d`; việc còn lại chỉ là chọn **giá nào** làm tròn lên.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 (số lần làm tròn lên bị ép).** Gọi `F = Σ floor(p_i)` và `k` là số giá có phần lẻ `> 0` (chỉ những giá này làm tròn lên mới **tăng** thêm `1`; giá nguyên có `floor = ceil`). Tổng đạt được nằm trong `[F, F + k]` và mọi giá trị nguyên trong đoạn đó đều đạt được. Do đó nếu `target < F` hoặc `target > F + k` thì `"-1"`. Ngược lại phải có **đúng** `d = target - F` giá (có phần lẻ `> 0`) làm tròn lên.

**Bước 2 (tham lam).** Một giá có phần lẻ `f` (đơn vị `0.001`) tốn `f` nếu xuống và `1000 - f` nếu lên. Nếu coi mọi giá đều làm tròn xuống thì sai số cơ sở là `S = Σ f`; mỗi giá làm tròn lên **thay đổi** sai số thêm `(1000 - f) - f = 1000 - 2f`. Cần chọn `d` giá để tổng các thay đổi này nhỏ nhất ⇒ chọn `d` giá có `1000 - 2f` nhỏ nhất ⇒ `f` lớn nhất.

**Đáp án:** `S + Σ_{d giá có f lớn nhất} (1000 - 2f)`, đổi sang chuỗi `3` chữ số thập phân.

| Cách | Ý tưởng |
|---|---|
| 2 | DP: `dp[c]` = sai số nhỏ nhất khi đã làm tròn lên đúng `c` giá |
| 3 | Sắp xếp phần lẻ giảm dần, `d` giá đầu làm tròn lên (nộp) |
| 4 | Đếm số giá theo từng phần lẻ `1..999`, lấy từ `999` xuống |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Quy hoạch động theo số lần làm tròn lên

**Giải thích từng đoạn.**

- Mỗi giá: tính `floor` (cộng vào `F`) và phần lẻ `f`.
- Giá có `f = 0`: không ảnh hưởng (cả hai cách làm tròn cho cùng số, sai số `0`).
- Giá có `f > 0`: `new[c] = min(dp[c] + f, dp[c - 1] + (1000 - f))`.
- Đáp án `dp[d]` với `d = target - F` nếu `0 <= d <= k`, ngược lại `"-1"`.

In [7]:
def parse_thousandths(p: str):
    whole, dec = p.split(".")
    return int(whole), int(dec)

def format_thousandths(err: int) -> str:
    return f"{err // 1000}.{err % 1000:03d}"

def minimize_dp(prices: List[str], target: int) -> str:
    floor_sum = 0
    fracs = []
    for p in prices:
        w, f = parse_thousandths(p)
        floor_sum += w
        if f:
            fracs.append(f)
    d = target - floor_sum
    if d < 0 or d > len(fracs):
        return "-1"
    INF = float("inf")
    dp = [0] + [INF] * len(fracs)                              # dp[c]: sai số nhỏ nhất (đơn vị 0.001) với đúng c giá làm tròn lên
    for idx, f in enumerate(fracs, 1):
        new = [INF] * (len(fracs) + 1)
        for c in range(idx + 1):
            if dp[c] != INF:
                new[c] = min(new[c], dp[c] + f)                # giá này làm tròn xuống
            if c >= 1 and dp[c - 1] != INF:
                new[c] = min(new[c], dp[c - 1] + 1000 - f)     # giá này làm tròn lên
        dp = new
    return format_thousandths(dp[d])

for prices, target, expected in examples:
    got = minimize_dp(prices, target)
    print(f"DP = {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

DP = '1.000' OK
DP = '-1' OK
DP = '1.500' OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def minimizeError(self, prices: List[str], target: int) -> str:
        floor_sum = 0
        fracs = []                                     # phần lẻ (đơn vị 0.001) của các giá có thể làm tròn lên
        for p in prices:
            whole, dec = p.split(".")
            floor_sum += int(whole)
            if int(dec):
                fracs.append(int(dec))
        d = target - floor_sum                         # số giá phải làm tròn lên
        k = len(fracs)
        if d < 0 or d > k:
            return "-1"
        INF = float("inf")
        dp = [0] + [INF] * k                           # dp[c]: sai số nhỏ nhất khi đúng c giá làm tròn lên
        for idx, f in enumerate(fracs, 1):
            new = [INF] * (k + 1)
            for c in range(idx + 1):
                if dp[c] != INF:
                    new[c] = dp[c] + f                 # làm tròn xuống tốn f
                if c >= 1 and dp[c - 1] != INF:
                    new[c] = min(new[c], dp[c - 1] + 1000 - f)   # làm tròn lên tốn 1000 - f
            dp = new
        err = dp[d]
        return f"{err // 1000}.{err % 1000:03d}"

sol_dp = lambda p, t: Solution().minimizeError(p, t)
for prices, target, expected in examples:
    assert sol_dp(prices, target) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(k^2)` với `k <= n <= 500` (khoảng `250000` phép), bộ nhớ `O(k)`.

---

### 7.2. Cách 3 — Tham lam: sắp xếp phần lẻ giảm dần (cách nộp chính)

**Giải thích từng đoạn.**

- Tách mỗi chuỗi thành phần nguyên (cộng vào `floor_sum`) và phần lẻ `f` (số nguyên đơn vị `0.001`); chỉ giữ phần lẻ `> 0`.
- `d = target - floor_sum`; nếu `d < 0` hoặc `d > len(fracs)` thì `"-1"`.
- Sắp xếp `fracs` giảm dần. `d` phần tử đầu làm tròn lên (tốn `1000 - f`), phần còn lại làm tròn xuống (tốn `f`).
- Cộng lại và định dạng.

In [9]:
def minimize_greedy(prices: List[str], target: int) -> str:
    floor_sum = 0
    fracs = []
    for p in prices:
        w, f = parse_thousandths(p)
        floor_sum += w
        if f:
            fracs.append(f)
    d = target - floor_sum
    if d < 0 or d > len(fracs):
        return "-1"
    fracs.sort(reverse=True)
    err = sum(1000 - f for f in fracs[:d]) + sum(fracs[d:])
    return format_thousandths(err)

for prices, target, expected in examples:
    got = minimize_greedy(prices, target)
    print(f"tham lam = {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

tham lam = '1.000' OK
tham lam = '-1' OK
tham lam = '1.500' OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def minimizeError(self, prices: List[str], target: int) -> str:
        floor_sum = 0
        fracs = []                                     # phần lẻ (đơn vị 0.001) của các giá có thể làm tròn lên
        for p in prices:
            whole, dec = p.split(".")
            floor_sum += int(whole)
            if int(dec):                               # giá nguyên: floor = ceil, không làm tròn lên được
                fracs.append(int(dec))
        d = target - floor_sum                         # số giá phải làm tròn lên
        if d < 0 or d > len(fracs):
            return "-1"
        fracs.sort(reverse=True)                       # phần lẻ lớn nhất làm tròn lên tốn ít nhất (1000 - f)
        err = sum(1000 - f for f in fracs[:d]) + sum(fracs[d:])
        return f"{err // 1000}.{err % 1000:03d}"

sol_greedy = lambda p, t: Solution().minimizeError(p, t)
for prices, target, expected in examples:
    assert sol_greedy(prices, target) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n log n)`, bộ nhớ `O(n)`.

---

### 7.3. Cách 4 — Đếm phần lẻ theo từng giá trị

**Ý tưởng khác.** Phần lẻ chỉ có `999` giá trị khác `0` (`1..999`), nên thay vì sắp xếp ta đếm `cnt[f]` và duyệt `f` từ `999` xuống `1`, lấy tối đa `cnt[f]` giá cho đến khi đủ `d`. Thời gian `O(n + 1000)`, không phụ thuộc `log n`.

In [11]:
def minimize_counting(prices: List[str], target: int) -> str:
    floor_sum = 0
    cnt = [0] * 1000
    k = 0
    for p in prices:
        w, f = parse_thousandths(p)
        floor_sum += w
        if f:
            cnt[f] += 1
            k += 1
    d = target - floor_sum
    if d < 0 or d > k:
        return "-1"
    err = 0
    need = d
    for f in range(999, 0, -1):
        up = min(cnt[f], need)                         # up giá có phần lẻ f làm tròn lên
        err += up * (1000 - f) + (cnt[f] - up) * f
        need -= up
    return format_thousandths(err)

for prices, target, expected in examples:
    got = minimize_counting(prices, target)
    print(f"đếm = {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

đếm = '1.000' OK
đếm = '-1' OK
đếm = '1.500' OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def minimizeError(self, prices: List[str], target: int) -> str:
        floor_sum = 0
        cnt = [0] * 1000                               # cnt[f]: số giá có phần lẻ f (đơn vị 0.001)
        k = 0
        for p in prices:
            whole, dec = p.split(".")
            floor_sum += int(whole)
            f = int(dec)
            if f:
                cnt[f] += 1
                k += 1
        d = target - floor_sum
        if d < 0 or d > k:
            return "-1"
        err = 0
        need = d                                       # còn cần làm tròn lên bao nhiêu giá
        for f in range(999, 0, -1):                    # phần lẻ lớn nhất trước
            up = min(cnt[f], need)
            err += up * (1000 - f) + (cnt[f] - up) * f
            need -= up
        return f"{err // 1000}.{err % 1000:03d}"

sol_count = lambda p, t: Solution().minimizeError(p, t)
for prices, target, expected in examples:
    assert sol_count(prices, target) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n + 1000)`, bộ nhớ `O(1000)`.

## 8. TRACE từng bước

### 8.1. Cách 3 cho ví dụ 1 và ví dụ 3

In `floor_sum`, các phần lẻ, `d`, các giá làm tròn lên và sai số từng phần.

In [13]:
def trace_greedy(prices, target):
    floor_sum, fracs = 0, []
    for p in prices:
        w, f = parse_thousandths(p)
        floor_sum += w
        print(f"  giá {p}: floor={w}, phần lẻ f={f}", "(giá nguyên, bỏ qua)" if f == 0 else "")
        if f:
            fracs.append(f)
    d = target - floor_sum
    print(f"floor_sum={floor_sum}, số giá có phần lẻ k={len(fracs)}, d = target - floor_sum = {d}")
    if d < 0 or d > len(fracs):
        print("ngoài khoảng [floor_sum, floor_sum + k] -> -1")
        return "-1"
    fracs.sort(reverse=True)
    up, down = fracs[:d], fracs[d:]
    print(f"phần lẻ giảm dần = {fracs}; làm tròn lên {up} (tốn {[1000 - f for f in up]}), làm tròn xuống {down} (tốn {down})")
    err = sum(1000 - f for f in up) + sum(down)
    print(f"sai số = {err} (đơn vị 0.001) -> {format_thousandths(err)}")
    return format_thousandths(err)

assert trace_greedy(["0.700", "2.800", "4.900"], 8) == "1.000"
print()
assert trace_greedy(["1.500", "2.500", "3.500"], 9) == "1.500"
print()
assert trace_greedy(["1.500", "2.500", "3.500"], 10) == "-1"

  giá 0.700: floor=0, phần lẻ f=700 
  giá 2.800: floor=2, phần lẻ f=800 
  giá 4.900: floor=4, phần lẻ f=900 
floor_sum=6, số giá có phần lẻ k=3, d = target - floor_sum = 2
phần lẻ giảm dần = [900, 800, 700]; làm tròn lên [900, 800] (tốn [100, 200]), làm tròn xuống [700] (tốn [700])
sai số = 1000 (đơn vị 0.001) -> 1.000

  giá 1.500: floor=1, phần lẻ f=500 
  giá 2.500: floor=2, phần lẻ f=500 
  giá 3.500: floor=3, phần lẻ f=500 
floor_sum=6, số giá có phần lẻ k=3, d = target - floor_sum = 3
phần lẻ giảm dần = [500, 500, 500]; làm tròn lên [500, 500, 500] (tốn [500, 500, 500]), làm tròn xuống [] (tốn [])
sai số = 1500 (đơn vị 0.001) -> 1.500

  giá 1.500: floor=1, phần lẻ f=500 
  giá 2.500: floor=2, phần lẻ f=500 
  giá 3.500: floor=3, phần lẻ f=500 
floor_sum=6, số giá có phần lẻ k=3, d = target - floor_sum = 4
ngoài khoảng [floor_sum, floor_sum + k] -> -1


### 8.2. Quy hoạch động cho ví dụ 1

In `dp` sau mỗi giá (chỉ các giá có phần lẻ).

In [14]:
fracs = [700, 800, 900]
INF = float("inf")
dp = [0] + [INF] * len(fracs)
print("ban đầu dp =", dp)
for idx, f in enumerate(fracs, 1):
    new = [INF] * (len(fracs) + 1)
    for c in range(idx + 1):
        if dp[c] != INF:
            new[c] = min(new[c], dp[c] + f)
        if c >= 1 and dp[c - 1] != INF:
            new[c] = min(new[c], dp[c - 1] + 1000 - f)
    dp = new
    print(f"sau giá f={f}: dp =", dp)
print("d = 2 -> dp[2] =", dp[2], "->", format_thousandths(dp[2]))
assert dp[2] == 1000

ban đầu dp = [0, inf, inf, inf]
sau giá f=700: dp = [700, 300, inf, inf]
sau giá f=800: dp = [1500, 900, 500, inf]
sau giá f=900: dp = [2400, 1600, 1000, 600]
d = 2 -> dp[2] = 1000 -> 1.000


### 8.3. Vì sao số thực dễ sai (minh họa)

Cộng các sai số bằng số thực rồi **cắt** về `3` chữ số cho kết quả sai ở ví dụ 1.

In [15]:
errs = [0.7 - 0, 3 - 2.8, 5 - 4.9]
total = sum(errs)
print(errs, "-> tổng =", repr(total))
print("cắt bớt chữ số:", f"{int(total * 1000) / 1000:.3f}", "| định dạng đúng bằng số nguyên:", format_thousandths(700 + 200 + 100))

[0.7, 0.20000000000000018, 0.09999999999999964] -> tổng = 0.9999999999999998
cắt bớt chữ số: 0.999 | định dạng đúng bằng số nguyên: 1.000


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (tham lam sắp xếp): ngắn, `O(n log n)`.

```python
class Solution:
    def minimizeError(self, prices: List[str], target: int) -> str:
        floor_sum = 0
        fracs = []                                     # phần lẻ (đơn vị 0.001) của các giá có thể làm tròn lên
        for p in prices:
            whole, dec = p.split(".")
            floor_sum += int(whole)
            if int(dec):                               # giá nguyên: floor = ceil, không làm tròn lên được
                fracs.append(int(dec))
        d = target - floor_sum                         # số giá phải làm tròn lên
        if d < 0 or d > len(fracs):
            return "-1"
        fracs.sort(reverse=True)                       # phần lẻ lớn nhất làm tròn lên tốn ít nhất (1000 - f)
        err = sum(1000 - f for f in fracs[:d]) + sum(fracs[d:])
        return f"{err // 1000}.{err % 1000:03d}"
```

In [16]:
class Solution:
    def minimizeError(self, prices: List[str], target: int) -> str:
        floor_sum = 0
        fracs = []                                     # phần lẻ (đơn vị 0.001) của các giá có thể làm tròn lên
        for p in prices:
            whole, dec = p.split(".")
            floor_sum += int(whole)
            if int(dec):                               # giá nguyên: floor = ceil, không làm tròn lên được
                fracs.append(int(dec))
        d = target - floor_sum                         # số giá phải làm tròn lên
        if d < 0 or d > len(fracs):
            return "-1"
        fracs.sort(reverse=True)                       # phần lẻ lớn nhất làm tròn lên tốn ít nhất (1000 - f)
        err = sum(1000 - f for f in fracs[:d]) + sum(fracs[d:])
        return f"{err // 1000}.{err % 1000:03d}"

sol = lambda p, t: Solution().minimizeError(p, t)
for prices, target, expected in examples:
    got = sol(prices, target)
    print(repr(got), "OK" if got == expected else "SAI")
    assert got == expected

'1.000' OK
'-1' OK
'1.500' OK


## 10. Vì sao đúng

**Tính khả thi.** Mỗi giá `p_i` có `floor_i` và `ceil_i`; `ceil_i - floor_i = 1` nếu có phần lẻ `> 0`, ngược lại `0`. Tổng làm tròn bằng `F + (số giá có phần lẻ > 0 làm tròn lên)`. Số này có thể là mọi giá trị trong `[0, k]`, nên `target` đạt được khi và chỉ khi `F <= target <= F + k`. Khi đó số giá phải làm tròn lên là đúng `d = target - F`.

**Tối ưu (hoán đổi).** Mỗi giá có phần lẻ `f` (đơn vị `0.001`) có sai số `f` nếu xuống và `1000 - f` nếu lên. Tổng sai số bằng `S + Σ_{i làm tròn lên} (1000 - 2 f_i)` với `S = Σ f_i`. `S` không đổi, nên cần chọn **đúng `d` giá** để tối thiểu hóa `Σ (1000 - 2 f_i)`, tức chọn `d` giá có `f_i` lớn nhất. Nếu một lời giải tối ưu có giá `a` làm tròn lên nhưng giá `b` có `f_b > f_a` làm tròn xuống thì đổi vai (`a` xuống, `b` lên) thay đổi sai số thêm `(1000 - 2 f_b) - (1000 - 2 f_a) = 2 (f_a - f_b) < 0`, tức tốt hơn: mâu thuẫn. Vậy tham lam (chọn `d` phần lẻ lớn nhất) tối ưu. Trường hợp bằng nhau thì kết quả tổng bằng nhau.

**Cách 2** là DP đúng định nghĩa nên đúng độc lập; **Cách 4** chỉ là Cách 3 với sắp xếp bằng đếm (duyệt `f` giảm dần, đúng thứ tự sắp xếp giảm dần).

**Độ chính xác.** Mọi phép tính dùng **số nguyên** (đơn vị `0.001`) nên không có sai số làm tròn; chuỗi kết quả `x // 1000` và `x % 1000` đệm `0` đủ `3` chữ số.

**Kiểm chứng bằng máy.** Oracle (Cách 1) tính bằng `Fraction` (chính xác tuyệt đối) trên mọi cách làm tròn. Ta đối chiếu mọi cách với oracle trên **mọi** đa tập giá nhỏ (phần nguyên `0..1`, phần lẻ trong tập đại diện) với **mọi** `target` từ dưới `F` đến trên `F + n`, cộng nhiều test ngẫu nhiên có seed.

In [17]:
alphabet = ["0.000", "0.100", "0.500", "0.900", "1.000", "1.100", "1.500", "1.900"]
total = 0
for n in range(1, 6):
    for prices in combinations_with_replacement(alphabet, n):
        prices = list(prices)
        F = sum(int(p.split(".")[0]) for p in prices)
        for target in range(max(0, F - 1), F + n + 2):
            ref = minimize_brute(prices, target)
            assert minimize_dp(prices, target) == minimize_greedy(prices, target) == minimize_counting(prices, target) == ref, (prices, target)
            assert sol_dp(prices, target) == sol_greedy(prices, target) == sol_count(prices, target) == sol(prices, target) == ref
            total += 1
print("đã đối chiếu", total, "cặp (prices, target): mọi cách khớp oracle Fraction")

đã đối chiếu 9453 cặp (prices, target): mọi cách khớp oracle Fraction


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi cách làm tròn | `O(n * 2^n)` | `O(n)` |
| 2. DP theo số lần làm tròn lên | `O(k^2)` | `O(k)` |
| 3. Tham lam sắp xếp | `O(n log n)` | `O(n)` |
| 4. Đếm phần lẻ | `O(n + 1000)` | `O(1000)` |

Đo ở cỡ tối đa (`n = 500`) với các dạng: ngẫu nhiên, toàn giá nguyên, toàn phần lẻ giống nhau, phần lẻ lớn (`.999`), giá tối đa `1000.000`:

In [18]:
rng = random.Random(11)
n = 500
def make(kind):
    if kind == "ngẫu nhiên":
        return [f"{rng.randint(0, 999)}.{rng.randint(0, 999):03d}" for _ in range(n)]
    if kind == "toàn giá nguyên":
        return [f"{rng.randint(0, 1000)}.000" for _ in range(n)]
    if kind == "phần lẻ giống nhau":
        return [f"{rng.randint(0, 1000)}.500" for _ in range(n)]
    if kind == "phần lẻ .999":
        return [f"{rng.randint(0, 999)}.999" for _ in range(n)]
    return ["1000.000"] * n
for kind in ("ngẫu nhiên", "toàn giá nguyên", "phần lẻ giống nhau", "phần lẻ .999", "giá tối đa"):
    prices = make(kind)
    F = sum(int(p.split(".")[0]) for p in prices)
    for target in (F, F + n // 2, F + n):
        ref = sol_greedy(prices, target)
        row = [f"{kind:22} target=F+{target - F:3}"]
        for name, fn in (("DP", sol_dp), ("tham lam", sol_greedy), ("đếm", sol_count)):
            t0 = time.perf_counter()
            got = fn(prices, target)
            dt = (time.perf_counter() - t0) * 1000
            assert got == ref, (kind, target, name, got, ref)
            row.append(f"{name} {dt:7.2f} ms")
        print(" | ".join(row), "| đáp án", ref)

ngẫu nhiên             target=F+  0 | DP    0.34 ms | tham lam    0.32 ms | đếm    0.35 ms | đáp án 248.926
ngẫu nhiên             target=F+250 | DP    0.33 ms | tham lam    0.41 ms | đếm    0.31 ms | đáp án 121.582
ngẫu nhiên             target=F+500 | DP    0.28 ms | tham lam    0.26 ms | đếm    0.24 ms | đáp án -1
toàn giá nguyên        target=F+  0 | DP    0.19 ms | tham lam    0.19 ms | đếm    0.22 ms | đáp án 0.000
toàn giá nguyên        target=F+250 | DP    0.20 ms | tham lam    0.17 ms | đếm    0.17 ms | đáp án -1
toàn giá nguyên        target=F+500 | DP    0.23 ms | tham lam    0.33 ms | đếm    0.20 ms | đáp án -1
phần lẻ giống nhau     target=F+  0 | DP    0.26 ms | tham lam    0.19 ms | đếm    0.22 ms | đáp án 250.000
phần lẻ giống nhau     target=F+250 | DP    0.31 ms | tham lam    0.27 ms | đếm    0.29 ms | đáp án 250.000
phần lẻ giống nhau     target=F+500 | DP    0.32 ms | tham lam    0.25 ms | đếm    0.24 ms | đáp án 250.000
phần lẻ .999           target=F+  0 | DP    0

## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle `Fraction`), kiểm tra bằng `Fraction`/`Decimal` độc lập, test ngẫu nhiên có seed và các tính chất.

In [19]:
def check_all(prices, target, expected=None, brute=True):
    results = {"DP": sol_dp(prices, target), "tham lam": sol_greedy(prices, target), "đếm": sol_count(prices, target), "nộp": sol(prices, target),
               "minimize_dp": minimize_dp(prices, target), "minimize_greedy": minimize_greedy(prices, target), "minimize_counting": minimize_counting(prices, target)}
    first = results["nộp"]
    for name, got in results.items():
        assert got == first, (name, got, first)
    if brute and len(prices) <= 12:
        assert minimize_brute(prices, target) == first, (prices, target)
    if expected is not None:
        assert first == expected, (prices, target, first, expected)
    return first

# 12.1 ví dụ đề
for p, t, e in examples:
    check_all(p, t, e)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (["0.000"], 0, "0.000"),                                           # giá nguyên, sai số 0
    (["0.000"], 1, "-1"),                                              # giá nguyên không thể làm tròn lên
    (["1.000", "2.000"], 3, "0.000"),                                  # mọi giá nguyên, đúng target
    (["1.000", "2.000"], 4, "-1"),                                     # không thể tăng
    (["1.000", "2.000"], 2, "-1"),                                     # target < tổng floor
    (["0.500"], 0, "0.500"),                                           # xuống: sai số 0.5
    (["0.500"], 1, "0.500"),                                           # lên: sai số 0.5
    (["999.999"], 1000, "0.001"),                                      # làm tròn lên: sai số rất nhỏ
    (["999.999"], 999, "0.999"),                                       # làm tròn xuống: sai số lớn
    (["0.001"] * 500, 0, "0.500"),                                     # xuống hết: 500 * 0.001
    (["0.001"] * 500, 500, "499.500"),                                 # lên hết: 500 * 0.999
    (["1000.000"] * 500, 500000, "0.000"),                             # cỡ tối đa, toàn nguyên
    (["1000.000"] * 500, 10 ** 6, "-1"),                               # target tối đa vượt khả năng
    (["0.500", "0.500"], 1, "1.000"),                                  # một lên, một xuống: 0.5 + 0.5
    (["0.200", "1.800", "2.500", "3.000"], 7, "0.900"),                # tính tay ở bài tập 2
    (["0.700", "2.800", "4.900"], 6, "2.400"),                         # xuống hết: 0.7 + 0.8 + 0.9
    (["0.700", "2.800", "4.900"], 9, "0.600"),                         # lên hết: 0.3 + 0.2 + 0.1
    (["0.700", "2.800", "4.900"], 5, "-1"),                            # dưới tổng floor
]
for p, t, e in edge:
    check_all(p, t, e)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 kiểm tra độc lập bằng Decimal: sai số của chính phương án tìm được bằng đáp án
def decimal_check(prices, target, answer):
    """Dựng một phương án làm tròn đạt đáp án (xuống/lên theo phần lẻ giảm dần) và tính lại sai số bằng Decimal."""
    ds = [Decimal(p) for p in prices]
    fl = [int(d.to_integral_value(rounding="ROUND_FLOOR")) for d in ds]
    cands = sorted([i for i, d in enumerate(ds) if d != fl[i]], key=lambda i: ds[i] - fl[i], reverse=True)
    need = target - sum(fl)
    if need < 0 or need > len(cands):
        return answer == "-1"
    up = set(cands[:need])
    err = sum((fl[i] + 1 - ds[i]) if i in up else (ds[i] - fl[i]) for i in range(len(ds)))
    return f"{err:.3f}" == answer

rng = random.Random(123)
for _ in range(300):
    prices = [f"{rng.randint(0, 20)}.{rng.randint(0, 999):03d}" for _ in range(rng.randint(1, 8))]
    F = sum(int(p.split(".")[0]) for p in prices)
    t = rng.randint(max(0, F - 1), F + len(prices) + 1)
    assert decimal_check(prices, t, sol(prices, t)), (prices, t)
print("12.3 kiểm tra Decimal: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 18 ca
12.3 kiểm tra Decimal: OK


In [20]:
# 12.4 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle Fraction)
def wrong_float_truncate(prices, target):          # dùng số thực rồi cắt bớt chữ số thay vì định dạng làm tròn
    ps = [float(p) for p in prices]
    fl = [int(x) for x in ps]
    fr = sorted([x - f for x, f in zip(ps, fl) if x != f], reverse=True)
    d = target - sum(fl)
    if d < 0 or d > len(fr):
        return "-1"
    err = sum(1 - x for x in fr[:d]) + sum(fr[d:])
    return f"{int(err * 1000) / 1000:.3f}"

def wrong_str_round(prices, target):               # str(round(.., 3)) cho "1.0" thay vì "1.000"
    res = sol(prices, target)
    return res if res == "-1" else str(round(float(res), 3))

def wrong_smallest_first(prices, target):          # làm tròn lên cho các phần lẻ NHỎ nhất
    floor_sum, fracs = 0, []
    for p in prices:
        w, f = parse_thousandths(p)
        floor_sum += w
        if f:
            fracs.append(f)
    d = target - floor_sum
    if d < 0 or d > len(fracs):
        return "-1"
    fracs.sort()
    err = sum(1000 - f for f in fracs[:d]) + sum(fracs[d:])
    return format_thousandths(err)

def wrong_count_all(prices, target):               # coi mọi giá (cả giá nguyên) đều làm tròn lên được: dùng n thay cho k
    floor_sum, fracs = 0, []
    for p in prices:
        w, f = parse_thousandths(p)
        floor_sum += w
        if f:
            fracs.append(f)
    d = target - floor_sum
    if d < 0 or d > len(prices):
        return "-1"
    fracs.sort(reverse=True)
    err = sum(1000 - f for f in fracs[:d]) + sum(fracs[d:])
    return format_thousandths(err)

def wrong_no_lower_bound(prices, target):          # chỉ kiểm tra cận trên, quên target < tổng floor
    floor_sum, fracs = 0, []
    for p in prices:
        w, f = parse_thousandths(p)
        floor_sum += w
        if f:
            fracs.append(f)
    d = target - floor_sum
    if d > len(fracs):
        return "-1"
    fracs.sort(reverse=True)
    err = sum(1000 - f for f in fracs[:d]) + sum(fracs[d:])
    return format_thousandths(err)

def find_counter(wrong, tries=30000, seed=9):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 6)
        prices = [f"{rng.randint(0, 5)}.{rng.choice([0, 100, 200, 500, 700, 800, 900, 999]):03d}" for _ in range(n)]
        F = sum(int(p.split(".")[0]) for p in prices)
        t = rng.randint(max(0, F - 2), F + n + 1)
        if wrong(prices, t) != minimize_brute(prices, t):
            return prices, t
    return None

counters = {}
for name, fn in (("số thực rồi cắt bớt chữ số", wrong_float_truncate), ("định dạng bằng str(round(...))", wrong_str_round),
                 ("làm tròn lên các phần lẻ nhỏ nhất", wrong_smallest_first), ("coi giá nguyên cũng làm tròn lên được", wrong_count_all),
                 ("thiếu kiểm tra target < tổng floor", wrong_no_lower_bound)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    p, t = case
    print(f"{name:40} prices={p} target={t}: sai -> {fn(p, t)!r}, đúng -> {minimize_brute(p, t)!r}")
    check_all(p, t)
print("12.4 phản ví dụ: OK")

số thực rồi cắt bớt chữ số               prices=['0.800'] target=1: sai -> '0.199', đúng -> '0.200'
định dạng bằng str(round(...))           prices=['3.500', '1.000', '5.800', '5.100', '2.100', '4.800'] target=25: sai -> '2.7', đúng -> '2.700'
làm tròn lên các phần lẻ nhỏ nhất        prices=['2.000', '3.100', '1.500', '3.700', '2.100'] target=13: sai -> '3.000', đúng -> '1.000'
coi giá nguyên cũng làm tròn lên được    prices=['4.000', '5.900', '1.999'] target=13: sai -> '0.101', đúng -> '-1'
thiếu kiểm tra target < tổng floor       prices=['4.800', '2.200', '1.000', '2.999'] target=8: sai -> '0.401', đúng -> '-1'
12.4 phản ví dụ: OK


In [21]:
# 12.5 ngẫu nhiên nhỏ: giá thập phân tổng quát, so oracle Fraction
rng = random.Random(4058)
for _ in range(1500):
    n = rng.randint(1, 9)
    prices = [f"{rng.randint(0, rng.choice([1, 5, 1000]))}.{rng.choice([0, rng.randint(0, 999), rng.randint(0, 999), 500]):03d}" for _ in range(n)]
    F = sum(int(p.split(".")[0]) for p in prices)
    t = rng.randint(max(0, F - 2), F + n + 2)
    check_all(prices, t)
print("12.5 ngẫu nhiên nhỏ: OK")

# 12.6 ngẫu nhiên lớn (n tới 500): so các cách nhanh với DP đúng định nghĩa
for _ in range(60):
    n = rng.randint(100, 500)
    prices = [f"{rng.randint(0, 1000 if rng.random() < 0.5 else 3)}.{rng.choice([0, rng.randint(0, 999)]):03d}" for _ in range(n)]
    prices = [p if float(p) <= 1000 else "1000.000" for p in prices]
    F = sum(int(p.split(".")[0]) for p in prices)
    t = rng.randint(max(0, F - 3), F + n + 3)
    check_all(prices, t, brute=False)
print("12.6 ngẫu nhiên lớn: OK")

# 12.7 tính chất: đáp án thay đổi lồi theo target; đối xứng floor/ceil: sai số(target = F) + sai số(target = F + k) = tổng 1 * k
for _ in range(300):
    n = rng.randint(1, 15)
    prices = [f"{rng.randint(0, 50)}.{rng.randint(0, 999):03d}" for _ in range(n)]
    F = sum(int(p.split(".")[0]) for p in prices)
    k = sum(1 for p in prices if p.split(".")[1] != "000")
    lo, hi = sol(prices, F), sol(prices, F + k)
    to_thousandths = lambda s: int(s.replace(".", ""))
    assert to_thousandths(lo) + to_thousandths(hi) == 1000 * k                 # tất cả xuống: Σ f; tất cả lên: Σ (1 - f)
    errs = [to_thousandths(sol(prices, F + d)) for d in range(k + 1)]
    diffs = [errs[i + 1] - errs[i] for i in range(k)]
    assert diffs == sorted(diffs)                                              # lồi: mỗi lần thêm 1 lần làm tròn lên tốn nhiều hơn lần trước
    assert sol(prices, F - 1) == "-1" and sol(prices, F + k + 1) == "-1"
print("12.7 tính chất: OK")

12.5 ngẫu nhiên nhỏ: OK
12.6 ngẫu nhiên lớn: OK
12.7 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng số thực rồi **cắt** bớt chữ số

`0.7 + 0.2 + 0.1` trong số thực là `0.9999999999999999`; cắt về `3` chữ số cho `"0.999"` thay vì `"1.000"`.

In [22]:
p, t = counters["số thực rồi cắt bớt chữ số"]
print("prices =", p, "target =", t, "| sai ->", wrong_float_truncate(p, t), "| đúng ->", sol(p, t))
assert wrong_float_truncate(p, t) != sol(p, t)
print("ví dụ 1 của đề với cách này ->", wrong_float_truncate(["0.700", "2.800", "4.900"], 8), "(đúng là '1.000')")

prices = ['0.800'] target = 1 | sai -> 0.199 | đúng -> 0.200
ví dụ 1 của đề với cách này -> 0.999 (đúng là '1.000')


Vì sao sai: số thực nhị phân không biểu diễn chính xác `0.1`, `0.2`, `0.7`, ...; cắt bớt chữ số làm lộ sai số. **Đúng:** đọc chuỗi thành số nguyên đơn vị `0.001`, tính toán bằng số nguyên, định dạng bằng `//` và `%`.

### Lỗi 2 — Định dạng bằng `str(round(x, 3))`

Đề yêu cầu **đúng** ba chữ số sau dấu thập phân.

In [23]:
p, t = counters["định dạng bằng str(round(...))"]
print("prices =", p, "target =", t, "| sai ->", repr(wrong_str_round(p, t)), "| đúng ->", repr(sol(p, t)))
assert wrong_str_round(p, t) != sol(p, t)

prices = ['3.500', '1.000', '5.800', '5.100', '2.100', '4.800'] target = 25 | sai -> '2.7' | đúng -> '2.700'


Vì sao sai: `str(round(1.0, 3))` là `"1.0"`. **Đúng:** `f"{err // 1000}.{err % 1000:03d}"` (hoặc `f"{x:.3f}"`).

### Lỗi 3 — Làm tròn lên cho các phần lẻ **nhỏ** nhất

In [24]:
p, t = counters["làm tròn lên các phần lẻ nhỏ nhất"]
print("prices =", p, "target =", t, "| sai ->", repr(wrong_smallest_first(p, t)), "| đúng ->", repr(sol(p, t)))
assert wrong_smallest_first(p, t) != sol(p, t)

prices = ['2.000', '3.100', '1.500', '3.700', '2.100'] target = 13 | sai -> '3.000' | đúng -> '1.000'


Vì sao sai: làm tròn lên một giá có phần lẻ `f` tốn `1 - f`, càng nhỏ khi `f` càng lớn. **Đúng:** sắp xếp **giảm dần** và lấy `d` phần tử đầu.

### Lỗi 4 — Coi giá nguyên cũng làm tròn lên được

Với giá nguyên `floor = ceil`, làm tròn "lên" không tăng tổng.

In [25]:
p, t = counters["coi giá nguyên cũng làm tròn lên được"]
print("prices =", p, "target =", t, "| sai ->", repr(wrong_count_all(p, t)), "| đúng ->", repr(sol(p, t)))
assert wrong_count_all(p, t) != sol(p, t)

prices = ['4.000', '5.900', '1.999'] target = 13 | sai -> '0.101' | đúng -> '-1'


Vì sao sai: cận trên của tổng là `floor_sum + k` với `k` = số giá **có phần lẻ**, không phải `n`. **Đúng:** chỉ đếm giá có phần lẻ `> 0`.

### Lỗi 5 — Thiếu kiểm tra `target < tổng floor`

Khi `d` âm, lát cắt `fracs[:d]` với `d` âm cho một đoạn vô nghĩa thay vì `"-1"`.

In [26]:
p, t = counters["thiếu kiểm tra target < tổng floor"]
print("prices =", p, "target =", t, "| sai ->", repr(wrong_no_lower_bound(p, t)), "| đúng ->", repr(sol(p, t)))
assert wrong_no_lower_bound(p, t) != sol(p, t) and sol(p, t) == "-1"

prices = ['4.800', '2.200', '1.000', '2.999'] target = 8 | sai -> '0.401' | đúng -> '-1'


Vì sao sai: không thể làm tổng nhỏ hơn tổng các `floor`. **Đúng:** `if d < 0 or d > k: return "-1"`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "làm tròn từng phần tử lên hoặc xuống để tổng bằng một giá trị cho trước, tối thiểu sai số" ⇒ số lần làm tròn lên bị **ép**; chọn phần tử theo **phần lẻ** (tham lam).
- Dữ liệu là số thực viết dưới dạng chuỗi với số chữ số thập phân cố định ⇒ **đổi sang số nguyên** (nhân `10^k`), tránh sai số nhị phân.
- Định dạng chuỗi kết quả bằng số nguyên: `f"{x // 1000}.{x % 1000:03d}"`.
- Muốn kiểm chứng: dùng `fractions.Fraction` hoặc `decimal.Decimal` làm oracle.
- Sắp xếp theo khóa nhỏ (`0..999`) có thể thay bằng **đếm** `O(n + 1000)`.
- Cùng họ: Minimize Rounding Error (nhóm bài làm tròn), Maximum Sum of Rounded (tham lam theo phần lẻ), Split Array Largest Sum (nhị phân + tham lam).
- Test nên có: giá toàn nguyên, `target` ngoài khoảng ở cả hai phía, phần lẻ `.999` / `.001` / `.500`, `n = 500`, `target = 10^6`.

## 15. Biến thể

1. **Số chữ số thập phân khác:** tổng quát hóa bằng `Fraction` (bội của `1 / 10^k`).
2. **Phương án cụ thể:** trả về danh sách giá đã làm tròn (bài tập 5).
3. **Trọng số:** mỗi giá có hệ số, sai số `Σ w_i |R_i - p_i|`: vẫn tham lam theo `w_i (1 - 2 f_i)`.
4. Bài liên quan: các bài làm tròn tổng có ràng buộc, "Minimize Rounding Error".

Kiểm biến thể 3 (có trọng số nguyên dương) bằng oracle `Fraction` thử mọi cách:

In [27]:
def weighted_greedy(prices: List[str], weights: List[int], target: int) -> str:
    floor_sum, items = 0, []
    base = Fraction(0)
    for p, w in zip(prices, weights):
        fr = Fraction(p)
        fl = fr.numerator // fr.denominator
        floor_sum += fl
        f = fr - fl
        base += w * f                                             # tất cả xuống
        if f:
            items.append(w * (1 - 2 * f))                         # thay đổi sai số khi làm tròn lên
    d = target - floor_sum
    if d < 0 or d > len(items):
        return "-1"
    items.sort()
    total = base + sum(items[:d])
    return str(total * 1000)                                      # đơn vị 0.001 để so sánh chính xác

def weighted_brute(prices, weights, target):
    ps = [Fraction(p) for p in prices]
    best = None
    for choice in product((0, 1), repeat=len(ps)):
        total, err = 0, Fraction(0)
        for p, w, c in zip(ps, weights, choice):
            r = math.ceil(p) if c else math.floor(p)
            total += r
            err += w * abs(r - p)
        if total == target and (best is None or err < best):
            best = err
    return "-1" if best is None else str(best * 1000)

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 7)
    prices = [f"{rng.randint(0, 5)}.{rng.randint(0, 999):03d}" for _ in range(n)]
    weights = [rng.randint(1, 5) for _ in range(n)]
    F = sum(int(p.split(".")[0]) for p in prices)
    t = rng.randint(max(0, F - 1), F + n + 1)
    assert weighted_greedy(prices, weights, t) == weighted_brute(prices, weights, t), (prices, weights, t)
print("biến thể có trọng số: tham lam theo w * (1 - 2f) khớp oracle Fraction")

biến thể có trọng số: tham lam theo w * (1 - 2f) khớp oracle Fraction


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cách | `Fraction`, `2^n` cách | `O(n 2^n)` | `O(n)` | Oracle |
| 2. DP | `dp[c]` = sai số nhỏ nhất với `c` lần lên | `O(k^2)` | `O(k)` | Khi tham lam khó chứng minh |
| 3. Tham lam sắp xếp | Phần lẻ lớn nhất làm tròn lên | `O(n log n)` | `O(n)` | **Cách nộp chính** |
| 4. Đếm | `cnt[f]`, `f` từ 999 xuống | `O(n + 1000)` | `O(1000)` | Nhanh nhất |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(parse_thousandths("12.034"), parse_thousandths("0.900"))
print(divmod(2750, 1000), f"{2750 // 1000}.{2750 % 1000:03d}", f"{9 // 1000}.{9 % 1000:03d}")
print(Solution().minimizeError(["0.700", "2.800", "4.900"], 6))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # cặp (phần nguyên, phần lẻ) của hai chuỗi
my_guess_2 = None      # divmod và hai chuỗi định dạng
my_guess_3 = None      # kết quả minimizeError với target = 6

### Bài 2 — Tính tay (dễ-vừa)

`prices = ["0.200", "1.800", "2.500", "3.000"]`, `target = 7`. Tính `floor_sum`, `d`, chọn giá làm tròn lên và ghi sai số.

In [29]:
# Bài 2: điền kết quả tính tay
hand_floor_sum, hand_d, hand_answer = None, None, None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách tham lam.

In [30]:
def exercise_3(prices: List[str], target: int) -> str:
    floor_sum, fracs = 0, []
    for p in prices:
        whole, dec = p.split(".")
        floor_sum += int(whole)
        # TODO 1: nếu int(dec) khác 0 thì thêm int(dec) vào fracs
    d = target - floor_sum
    # TODO 2: nếu d < 0 hoặc d > len(fracs) thì trả về "-1"
    # TODO 3: sắp xếp fracs giảm dần; d phần tử đầu tốn 1000 - f, còn lại tốn f; ghép chuỗi 3 chữ số thập phân
    pass


# for p, t, e in examples:
#     assert exercise_3(p, t) == e

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(prices, target)` dùng **đếm** phần lẻ (`cnt[f]`).

In [31]:
def exercise_4(prices: List[str], target: int) -> str:
    # TODO
    pass


# for p, t, e in examples:
#     assert exercise_4(p, t) == e

### Bài 5 — Mini challenge: phương án làm tròn cụ thể (khó)

Viết `exercise_5(prices, target)` trả về **danh sách các số nguyên đã làm tròn** (cùng thứ tự với `prices`) sao cho tổng bằng `target` và sai số nhỏ nhất; trả về `None` nếu không thể. Có thể có nhiều đáp án đúng nên kiểm bằng hàm `check_rounding`.

```python
exercise_5(["0.700","2.800","4.900"], 8) -> [0, 3, 5]
exercise_5(["1.500","2.500","3.500"], 10) -> None
```
Gợi ý: sắp xếp **chỉ số** theo phần lẻ giảm dần và đánh dấu `d` chỉ số đầu là làm tròn lên (chỉ cho giá có phần lẻ `> 0`).

In [32]:
def exercise_5(prices: List[str], target: int) -> Optional[List[int]]:
    # TODO
    pass


def check_rounding(prices: List[str], target: int, rounded) -> bool:
    best = minimize_brute(prices, target) if len(prices) <= 12 else sol(prices, target)
    if best == "-1":
        return rounded is None
    if rounded is None or len(rounded) != len(prices) or sum(rounded) != target:
        return False
    ps = [Fraction(p) for p in prices]
    if any(r not in (math.floor(p), math.ceil(p)) for r, p in zip(rounded, ps)):
        return False
    return fmt(sum(abs(r - p) for r, p in zip(rounded, ps))) == best


# for p, t, e in examples:
#     assert check_rounding(p, t, exercise_5(p, t))

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `parse_thousandths("a.bcd")` trả về `(int("a"), int("bcd"))`.
- Mức 2: `"0.900"` cho `(0, 900)`; `:03d` đệm số `0` bên trái.
- Mức 3: `(12, 34) (0, 900)`; `(2, 750) '2.750' '0.009'`; `'2.400'` (xuống hết: `0.7 + 0.8 + 0.9`).

**Bài 2**
- Mức 1: `floor_sum = 0 + 1 + 2 + 3`.
- Mức 2: `d = 7 - 6 = 1`; phần lẻ có thể làm tròn lên: `200, 800, 500` (giá `3.000` là số nguyên).
- Mức 3: làm tròn lên giá có phần lẻ lớn nhất `800`: tốn `200`; hai giá kia làm tròn xuống tốn `200 + 500`; tổng `900` ⇒ `"0.900"`.

**Bài 3**
- Mức 1: `fracs.append(int(dec))` khi `int(dec)` khác `0`.
- Mức 2: `if d < 0 or d > len(fracs): return "-1"`.
- Mức 3: `fracs.sort(reverse=True)`; `err = sum(1000 - f for f in fracs[:d]) + sum(fracs[d:])`; `return f"{err // 1000}.{err % 1000:03d}"`.

**Bài 4**
- Mức 1: `cnt = [0] * 1000`; `cnt[f] += 1` cho phần lẻ `f > 0`.
- Mức 2: duyệt `f` từ `999` xuống `1`; `up = min(cnt[f], need)`.
- Mức 3: `err += up * (1000 - f) + (cnt[f] - up) * f`; `need -= up`.

**Bài 5**
- Mức 1: tính `floor` của từng giá; `rounded = floors[:]`.
- Mức 2: danh sách chỉ số có phần lẻ `> 0`, sắp theo phần lẻ giảm dần.
- Mức 3: với `d` chỉ số đầu `rounded[i] += 1`; trả `None` nếu `d` ngoài `[0, k]`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
r1 = (parse_thousandths("12.034"), parse_thousandths("0.900"))
r2 = (divmod(2750, 1000), f"{2750 // 1000}.{2750 % 1000:03d}", f"{9 // 1000}.{9 % 1000:03d}")
r3 = Solution().minimizeError(["0.700", "2.800", "4.900"], 6)
print(r1, r2, r3)
assert r1 == ((12, 34), (0, 900))
assert r2 == ((2, 750), "2.750", "0.009")
assert r3 == "2.400"

((12, 34), (0, 900)) ((2, 750), '2.750', '0.009') 2.400


### Đáp án Bài 2

`floor_sum = 0 + 1 + 2 + 3 = 6`, `d = 7 - 6 = 1`. Các phần lẻ khác `0`: `200`, `800`, `500`. Làm tròn lên giá có phần lẻ `800` (tốn `200`), hai giá còn lại làm tròn xuống (tốn `200` và `500`); `3.000` là số nguyên nên không tốn. Tổng `900` (đơn vị `0.001`) ⇒ `"0.900"`.

In [34]:
hand_floor_sum_solution, hand_d_solution, hand_answer_solution = 6, 1, "0.900"
prices = ["0.200", "1.800", "2.500", "3.000"]
assert hand_floor_sum_solution == sum(int(p.split(".")[0]) for p in prices)
assert hand_d_solution == 7 - hand_floor_sum_solution
assert hand_answer_solution == sol(prices, 7) == minimize_brute(prices, 7)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(prices: List[str], target: int) -> str:
    floor_sum, fracs = 0, []
    for p in prices:
        whole, dec = p.split(".")
        floor_sum += int(whole)
        if int(dec):
            fracs.append(int(dec))
    d = target - floor_sum
    if d < 0 or d > len(fracs):
        return "-1"
    fracs.sort(reverse=True)
    err = sum(1000 - f for f in fracs[:d]) + sum(fracs[d:])
    return f"{err // 1000}.{err % 1000:03d}"

for p, t, e in examples:
    assert exercise_3_solution(p, t) == e
rng = random.Random(3)
for _ in range(500):
    n = rng.randint(1, 8)
    prices = [f"{rng.randint(0, 5)}.{rng.randint(0, 999):03d}" for _ in range(n)]
    F = sum(int(x.split(".")[0]) for x in prices)
    t = rng.randint(max(0, F - 1), F + n + 1)
    assert exercise_3_solution(prices, t) == minimize_brute(prices, t)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(prices: List[str], target: int) -> str:
    floor_sum, cnt, k = 0, [0] * 1000, 0
    for p in prices:
        whole, dec = p.split(".")
        floor_sum += int(whole)
        if int(dec):
            cnt[int(dec)] += 1
            k += 1
    d = target - floor_sum
    if d < 0 or d > k:
        return "-1"
    err, need = 0, d
    for f in range(999, 0, -1):
        up = min(cnt[f], need)
        err += up * (1000 - f) + (cnt[f] - up) * f
        need -= up
    return f"{err // 1000}.{err % 1000:03d}"

for p, t, e in examples:
    assert exercise_4_solution(p, t) == e
rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 8)
    prices = [f"{rng.randint(0, 5)}.{rng.randint(0, 999):03d}" for _ in range(n)]
    F = sum(int(x.split(".")[0]) for x in prices)
    t = rng.randint(max(0, F - 1), F + n + 1)
    assert exercise_4_solution(prices, t) == minimize_brute(prices, t)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(prices: List[str], target: int) -> Optional[List[int]]:
    floors, order = [], []
    for i, p in enumerate(prices):
        whole, dec = p.split(".")
        floors.append(int(whole))
        if int(dec):
            order.append((int(dec), i))
    d = target - sum(floors)
    if d < 0 or d > len(order):
        return None
    order.sort(reverse=True)
    rounded = floors[:]
    for _, i in order[:d]:
        rounded[i] += 1
    return rounded

def check_rounding_solution(prices, target, rounded):
    best = minimize_brute(prices, target) if len(prices) <= 12 else sol(prices, target)
    if best == "-1":
        return rounded is None
    if rounded is None or len(rounded) != len(prices) or sum(rounded) != target:
        return False
    ps = [Fraction(p) for p in prices]
    if any(r not in (math.floor(p), math.ceil(p)) for r, p in zip(rounded, ps)):
        return False
    return fmt(sum(abs(r - p) for r, p in zip(rounded, ps))) == best

for p, t, e in examples:
    assert check_rounding_solution(p, t, exercise_5_solution(p, t))
assert exercise_5_solution(["0.700", "2.800", "4.900"], 8) == [0, 3, 5]
assert exercise_5_solution(["1.500", "2.500", "3.500"], 10) is None
rng = random.Random(5)
for _ in range(500):
    n = rng.randint(1, 8)
    prices = [f"{rng.randint(0, 5)}.{rng.randint(0, 999):03d}" for _ in range(n)]
    F = sum(int(x.split(".")[0]) for x in prices)
    t = rng.randint(max(0, F - 1), F + n + 1)
    assert check_rounding_solution(prices, t, exercise_5_solution(prices, t)), (prices, t)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (tham lam theo phần lẻ, dùng số nguyên)

```python
class Solution:
    def minimizeError(self, prices: List[str], target: int) -> str:
        floor_sum = 0
        fracs = []                                     # phần lẻ (đơn vị 0.001), chỉ các giá có phần lẻ > 0
        for p in prices:
            whole, dec = p.split(".")
            floor_sum += int(whole)
            if int(dec):
                fracs.append(int(dec))
        d = target - floor_sum                         # số giá phải làm tròn lên
        if d < 0 or d > len(fracs):
            return "-1"
        fracs.sort(reverse=True)
        err = sum(1000 - f for f in fracs[:d]) + sum(fracs[d:])
        return f"{err // 1000}.{err % 1000:03d}"
```

Công thức nhớ nhanh: **`d = target - Σ floor`; làm tròn lên `d` giá có phần lẻ lớn nhất; sai số = `Σ(1000 - f)` cho `d` giá đó `+ Σ f` cho phần còn lại**.

### Checklist trước khi submit

- [ ] Dùng **số nguyên** (đơn vị `0.001`), không dùng `float`?
- [ ] Chỉ đếm giá có phần lẻ `> 0` là "có thể làm tròn lên" (`k`)?
- [ ] Kiểm tra cả hai phía: `d < 0` và `d > k` đều trả `"-1"`?
- [ ] Làm tròn lên cho các phần lẻ **lớn nhất** (sắp xếp giảm dần)?
- [ ] Chuỗi kết quả có đúng `3` chữ số thập phân (`:03d`)?
- [ ] Đã thử: giá toàn nguyên, `target` ngoài khoảng, `.999`, `.001`, `n = 500`, `target = 10^6`?